In [0]:
from pyspark.sql.functions import current_timestamp, col

def add_bronze_metadata(df):
    """
    Standardized lineage for Bronze (Unity Catalog Compliant).
    Extracts file path from the UC-native _metadata struct.
    """
    return df.withColumn("_ingested_at", current_timestamp()).withColumn(
        "_source_file", col("_metadata.file_path")
    ) 

In [0]:
from pyspark.sql.functions import current_timestamp, col

def apply_silver_metadata(df):
    """
    Injects Silver update timestamps and purges transient Bronze/CDF metadata.
    """
    transient_cols = [
        "_change_type",
        "_commit_version",
        "_commit_timestamp",
        "_rescued_data",
    ]

    # Reassign back to 'df' to ensure the variable always exists for the return statement
    if "_commit_version" in df.columns:
        df = df.withColumn("_bronze_version", col("_commit_version"))

    # Chained execution with direct unpack
    return df.withColumn("_updated_at", current_timestamp()).drop(*transient_cols)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

def add_ref_metadata(df):
    """
    Standardizes technical metadata for reference (lookup) tables.
    """
    return (df
        .withColumn("_updated_at", current_timestamp())        
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_processed_by", lit("lending_risk_ingest_job"))
    )